# Июнь–август: какая таблица точнее покрывает `commission_monthly`

Эталон — отчёты эквайринга за **три летних месяца**, колонка **«Комиссия (₽ в месяц)»**.

| Месяц | Файл |
|---|---|
| 2026-06 | `06_Июнь_2026.xlsx` |
| 2026-07 | `07_Июль_2026.xlsx` |
| 2026-08 | `08_Август_2026.xlsx` |

| Источник | Поле |
|---|---|
| **MPOS_RENT** | `n_amt` |
| **ЦФТ DOG_OPER** | `o.c_calc_summ` (все виды, без фильтра) |

По каждому месяцу и в сумме за лето:
1. **Сходится** с отчётом (оба ≠ 0 или оба пусто/0; сумма на пересечении ненулей).
2. **Не сходится:** источник ≠ 0 при отчёте 0; отчёт ≠ 0 при пустом/нулевом источнике.

В конце **VERDICT** по месяцам и за июнь–август.

Новый kernel нормален. Нужны Impala и три Excel в `/home/jovyan/documents/Equaring/Data`.  
Июньские кадры из прошлой сверки **не** переиспользуются — окно другое.


In [ ]:
import re
from decimal import Decimal, InvalidOperation
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from rail_connectors.connection import connect

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 140)
pd.set_option('display.width', 220)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}'.replace(',', ' '))

DATA_DIR = Path('/home/jovyan/documents/Equaring/Data')
OUT_DIR = DATA_DIR / 'qc_summer_commission_monthly_winner'
OUT_DIR.mkdir(parents=True, exist_ok=True)

FOCUS_MONTHS = ['2026-06', '2026-07', '2026-08']
PERIOD_START = '2026-06-01'
PERIOD_END = '2026-08-31'
PERIOD_END_EXCL = '2026-09-01'
VAT = 1.22
EPS = 0.01
MEM_LIMIT = '8g'
NOTEBOOK_REV = '2026-09-18-summer-v1'

EXCEL_FILES = {
    '2026-06': {'names': ['06_Июнь_2026.xlsx'], 'header': 0},
    '2026-07': {'names': ['07_Июль_2026.xlsx'], 'header': -1},
    '2026-08': {'names': ['08_Август_2026.xlsx', '08_август_2026.xlsx'], 'header': -1},
}
MONTHLY_ALIASES = [
    'Комиссия (₽ в месяц)', 'Комиссия (руб в месяц)', 'Комиссия в месяц',
    'Комиссия CN (₽ в месяц)', 'commission_monthly', 'Комиссия \n(₽ в месяц)',
]

print('NOTEBOOK_REV:', NOTEBOOK_REV)
print('FOCUS_MONTHS:', FOCUS_MONTHS)
print('OUT_DIR:', OUT_DIR)
print('Эталон = отчёты июня–августа | MPOS = n_amt | ЦФТ = o.c_calc_summ')


## 0) Helpers + Impala


In [ ]:
def normalize_inn_q1(v):
    if pd.isna(v):
        return None
    s = str(v).strip()
    s = re.sub(r'\.0$', '', s)
    s = re.sub(r'\D+', '', s)
    if not s:
        return None
    if len(s) == 9:
        s = s.zfill(10)
    elif len(s) == 11:
        s = s.zfill(12)
    return s if len(s) in (10, 12) else None


def normalize_agr_q1(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace('\xa0', '').replace(' ', '').replace(',', '.')
    if s in {'', 'nan', 'None'}:
        return None
    try:
        d = Decimal(s)
        if d == d.to_integral_value():
            return str(int(d))
    except (InvalidOperation, ValueError):
        pass
    s = re.sub(r'\.0$', '', s)
    return s if s not in {'', 'nan', 'None'} else None


def to_num(s):
    return pd.to_numeric(
        s.astype(str).str.replace('\xa0', '', regex=False).str.replace(' ', '', regex=False).str.replace(',', '.', regex=False),
        errors='coerce',
    )


def pick_col(columns, aliases):
    cols = list(columns)
    norm = lambda x: re.sub(r'\s+', ' ', str(x).replace('\xa0', ' ').replace('\n', ' ').strip().lower())
    nmap = {norm(c): c for c in cols}
    for a in aliases:
        if a in cols:
            return a
        if norm(a) in nmap:
            return nmap[norm(a)]
    for a in aliases:
        key = norm(a)
        for nk, orig in nmap.items():
            if key and key in nk:
                return orig
    return None


def fetch_imp(sql, label):
    t0 = pd.Timestamp.now()
    print(f'FETCH {label} ...')
    with imp:
        imp.execute(f'set MEM_LIMIT={MEM_LIMIT}')
        df = imp.fetch(sql)
    if df is None:
        df = pd.DataFrame()
    print(f'  rows={len(df):,}  {(pd.Timestamp.now() - t0).total_seconds():.1f}s')
    return df


def is_nz(s):
    return pd.to_numeric(s, errors='coerce').fillna(0).abs() >= EPS


def resolve_excel_path(month):
    spec = EXCEL_FILES[month]
    for name in spec['names']:
        p = DATA_DIR / name
        if p.exists():
            return p
    hits = sorted(DATA_DIR.glob(f'*{month[-2:]}*2026*.xlsx')) + sorted(DATA_DIR.glob('*вгуст*.xlsx'))
    if hits:
        print(f'  fallback Excel {month}: {hits[0].name}')
        return hits[0]
    return None


def read_excel_report(path, header_hint):
    tried = []
    headers = list(range(0, 6)) if header_hint == -1 else [header_hint, *range(0, 6)]
    seen = []
    for h in headers:
        if h in seen:
            continue
        seen.append(h)
        raw = pd.read_excel(path, header=h)
        col_inn = pick_col(raw.columns, ['ИНН', 'inn', 'c_inn'])
        col_agr = pick_col(raw.columns, ['ID договора', 'agr_id', 'abs_agr_id', 'ИД договора'])
        col_mon = pick_col(raw.columns, MONTHLY_ALIASES)
        tried.append((h, col_inn, col_agr, col_mon, list(raw.columns)[:12]))
        if col_inn and col_agr and col_mon:
            return raw, h, col_inn, col_agr, col_mon
    raise RuntimeError(f'Не нашли колонки в {path.name}. tried={tried}')


if 'imp' in globals() and imp is not None:
    print('Reuse existing imp')
else:
    imp = connect(
        to='IMPALA',
        extra_options={'db': 'sandbox_ai'},
        driver_args={'tez.queue.name': 'ai'},
        kerberos={
            'keytab_path': '/home/jovyan/test_requests/tech.keytab',
            'use_credentials': True,
            'update_keytab': True,
        },
        user_params={'user_name': 'Shestopalov-VYur'},
    )
    imp._init_connection()
    print('Impala connected')


## 1) Эталон: отчёты июня, июля, августа


In [ ]:
excel_parts = []
excel_meta = []
missing = []
for month in FOCUS_MONTHS:
    path = resolve_excel_path(month)
    if path is None:
        missing.append(month)
        print('MISSING Excel for', month, '| DATA_DIR xlsx:', [p.name for p in DATA_DIR.glob('*.xlsx')][:30])
        continue
    header_hint = EXCEL_FILES[month]['header']
    raw_ex, hdr, col_inn, col_agr, col_mon = read_excel_report(path, header_hint)
    col_tar = pick_col(raw_ex.columns, ['Тариф', 'Тарифный план', 'tariff_name'])
    print(f'{month}: {path.name} header={hdr} inn={col_inn} agr={col_agr} monthly={col_mon} rows={len(raw_ex):,}')
    part = pd.DataFrame({
        'report_month': month,
        'inn_key': raw_ex[col_inn].map(normalize_inn_q1),
        'agr_id_key': raw_ex[col_agr].map(normalize_agr_q1),
        'commission_excel': to_num(raw_ex[col_mon]),
        'tariff': raw_ex[col_tar] if col_tar else np.nan,
    })
    part = (
        part.dropna(subset=['agr_id_key'])
        .groupby(['report_month', 'inn_key', 'agr_id_key'], as_index=False)
        .agg(commission_excel=('commission_excel', 'max'), tariff=('tariff', 'first'))
    )
    part['commission_excel'] = pd.to_numeric(part['commission_excel'], errors='coerce').fillna(0)
    part['excel_nz'] = is_nz(part['commission_excel'])
    excel_parts.append(part)
    excel_meta.append({
        'report_month': month,
        'file': path.name,
        'header': hdr,
        'keys': len(part),
        'excel_nz': int(part['excel_nz'].sum()),
        'excel_zero': int((~part['excel_nz']).sum()),
        'sum_nz': float(part.loc[part['excel_nz'], 'commission_excel'].sum()),
    })

if missing:
    raise FileNotFoundError(
        f'Нет отчётов за {missing}. Положите файлы в {DATA_DIR} '
        f'(06_Июнь_2026.xlsx, 07_Июль_2026.xlsx, 08_Август_2026.xlsx) и перезапустите.'
    )

ex_all = pd.concat(excel_parts, ignore_index=True)
excel_stat = pd.DataFrame(excel_meta)
excel_stat['excel_nz_pct'] = (100.0 * excel_stat['excel_nz'] / excel_stat['keys']).round(2)
print('\\n=== Эталон Excel июнь–август ===')
display(excel_stat)
print('keys all months=', f'{len(ex_all):,}', '| unique agr=', f'{ex_all["agr_id_key"].nunique():,}')


## 2) MPOS_RENT за июнь–август


In [ ]:
sql_mpos = f'''
with rent_base as (
  select
    cast(c_nmrc as string) as c_nmrc,
    cast(d_rent as date) as d_rent_dt,
    cast(n_amt as double) as n_amt_num,
    cast(ods_commit_ts as timestamp) as ods_commit_ts,
    cast(ods_insert_ts as timestamp) as ods_insert_ts,
    cast(ods_op_csn as decimal(38, 0)) as ods_op_csn,
    coalesce(cast(ods_deleted_flg as string), '0') as ods_deleted_flg
  from ods_alpha.scd1_mrc_pos_rent
  where c_nmrc is not null
    and cast(d_rent as date) between cast('{PERIOD_START}' as date) and cast('{PERIOD_END}' as date)
),
rent_ranked as (
  select *,
    row_number() over (
      partition by c_nmrc, d_rent_dt
      order by coalesce(ods_commit_ts, ods_insert_ts) desc, ods_op_csn desc, ods_insert_ts desc
    ) as rn
  from rent_base
  where ods_deleted_flg not in ('1', 'Y', 'y')
),
rent_dedup as (
  select c_nmrc, d_rent_dt, n_amt_num from rent_ranked where rn = 1
),
terms_active as (
  select distinct
    cast(t.n_agr as string) as n_agr,
    cast(t.c_nmrc as string) as c_nmrc,
    cast(t.d_valid_from as date) as d_valid_from,
    cast(t.d_valid_to as date) as d_valid_to
  from ods_alpha.scd1_agr_terms t
  where coalesce(cast(t.ods_deleted_flg as string), '0') not in ('1', 'Y', 'y')
    and t.c_nmrc is not null
    and cast(t.d_valid_from as date) <= cast('{PERIOD_END}' as date)
    and (t.d_valid_to is null or cast(t.d_valid_to as date) >= cast('{PERIOD_START}' as date))
),
agreements_active as (
  select distinct
    cast(a.n_agr as string) as n_agr,
    cast(a.abs_agr_id as string) as agr_id,
    cast(a.n_cmp_client as string) as n_cmp_client,
    cast(a.d_valid_from as date) as d_valid_from,
    cast(a.d_valid_to as date) as d_valid_to
  from ods_alpha.scd1_agreements a
  where coalesce(cast(a.ods_deleted_flg as string), '0') not in ('1', 'Y', 'y')
    and upper(trim(cast(a.acq_class as string))) = 'SA'
    and cast(a.d_valid_from as date) <= cast('{PERIOD_END}' as date)
    and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{PERIOD_START}' as date))
),
companies_active as (
  select distinct
    cast(c.n_cmp as string) as n_cmp,
    regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') as inn_key
  from ods_alpha.scd1_companies c
  where coalesce(cast(c.ods_deleted_flg as string), '0') not in ('1', 'Y', 'y')
    and c.c_inn is not null
)
select
  substr(cast(r.d_rent_dt as string), 1, 7) as report_month,
  c.inn_key as inn,
  cast(a.agr_id as string) as agr_id,
  count(*) as rent_rows,
  sum(r.n_amt_num) as commission_mpos
from rent_dedup r
left join terms_active t
  on t.c_nmrc = r.c_nmrc
 and r.d_rent_dt between t.d_valid_from and coalesce(t.d_valid_to, cast('2999-12-31' as date))
left join agreements_active a
  on a.n_agr = t.n_agr
 and r.d_rent_dt between a.d_valid_from and coalesce(a.d_valid_to, cast('2999-12-31' as date))
left join companies_active c
  on c.n_cmp = a.n_cmp_client
group by 1, 2, 3
'''

mpos_raw = fetch_imp(sql_mpos, 'MPOS mapped Jun-Aug')
mpos = mpos_raw.copy()
mpos['report_month'] = mpos['report_month'].astype(str).str[:7]
mpos['inn_key'] = mpos['inn'].map(normalize_inn_q1)
mpos['agr_id_key'] = mpos['agr_id'].map(normalize_agr_q1)
mpos['commission_mpos'] = pd.to_numeric(mpos['commission_mpos'], errors='coerce')
mpos_map = (
    mpos.dropna(subset=['agr_id_key'])
    .query('report_month in @FOCUS_MONTHS')
    .groupby(['report_month', 'inn_key', 'agr_id_key'], as_index=False)
    .agg(commission_mpos=('commission_mpos', 'sum'), rent_rows=('rent_rows', 'sum'))
)
mpos_map['mpos_nz'] = is_nz(mpos_map['commission_mpos'])
print('MPOS keys=', f'{len(mpos_map):,}')
display(
    mpos_map.groupby('report_month', as_index=False)
    .agg(keys=('agr_id_key', 'nunique'), nz=('mpos_nz', 'sum'), sum_mpos=('commission_mpos', 'sum'))
    .sort_values('report_month')
)


## 3) ЦФТ: `sum(o.c_calc_summ)` за июнь–август


In [ ]:
sql_cft = f'''
select
  regexp_replace(trim(cast(cl.c_inn as string)), '[^0-9]', '') as inn,
  cast(m.id as string) as agr_id,
  cast(vc.c_name as string) as commis_type,
  cast(o.c_date_create as timestamp) as c_date_create,
  cast(o.c_pay_summ as double) as c_pay_summ,
  cast(o.c_calc_summ as double) as c_calc_summ
from ods.scd1_z_R2_IP_DOG_OPER o
join ods.scd1_z_R2_VID_COMISS vc on vc.id = o.c_vid_comiss
join ods.scd1_z_r2_ip_merchants m on m.id = o.c_parent_id
join ods.scd1_z_client cl on m.c_cl_org = cl.id
where o.c_parent_class = 'R2_IP_MERCHANTS'
  and cl.class_id = 'CL_ORG'
  and cast(o.c_date_create as date) >= cast('{PERIOD_START}' as date)
  and cast(o.c_date_create as date) < cast('{PERIOD_END_EXCL}' as date)
'''
try:
    cft_raw = fetch_imp(sql_cft, 'CFT DOG_OPER Jun-Aug')
except Exception as exc:
    print('mixed-case fail, try lowercase:', type(exc).__name__, exc)
    sql_cft = sql_cft.replace('scd1_z_R2_IP_DOG_OPER', 'scd1_z_r2_ip_dog_oper').replace('scd1_z_R2_VID_COMISS', 'scd1_z_r2_vid_comiss')
    cft_raw = fetch_imp(sql_cft, 'CFT DOG_OPER Jun-Aug lc')

cft = cft_raw.copy()
cft['inn_key'] = cft['inn'].map(normalize_inn_q1)
cft['agr_id_key'] = cft['agr_id'].map(normalize_agr_q1)
cft['c_date_create'] = pd.to_datetime(cft['c_date_create'], errors='coerce')
cft['report_month'] = cft['c_date_create'].dt.strftime('%Y-%m')
cft['c_calc_summ'] = pd.to_numeric(cft['c_calc_summ'], errors='coerce')
cft['c_pay_summ'] = pd.to_numeric(cft['c_pay_summ'], errors='coerce')
cft = cft.dropna(subset=['agr_id_key'])
cft = cft.loc[cft['report_month'].isin(FOCUS_MONTHS)].copy()

cft_map = (
    cft.groupby(['report_month', 'inn_key', 'agr_id_key'], as_index=False)
    .agg(
        cft_rows=('c_calc_summ', 'size'),
        commission_monthly_cft=('c_calc_summ', 'sum'),
        commission_cft_pay=('c_pay_summ', 'sum'),
    )
)
cft_map['cft_nz'] = is_nz(cft_map['commission_monthly_cft'])
print('CFT monthly = sum(c_calc_summ), no type filter | rows=', f'{len(cft):,}')
display(
    cft_map.groupby('report_month', as_index=False)
    .agg(keys=('agr_id_key', 'nunique'), nz=('cft_nz', 'sum'), sum_calc=('commission_monthly_cft', 'sum'))
    .sort_values('report_month')
)


## 4) Счётчики по месяцам

На периметре каждого отчёта.

| Бакет | Смысл |
|---|---|
| оба ≠ 0 | отчёт и источник заполнены |
| оба 0 / нет строки | отчёт 0 и в источнике нет ненуля |
| источник ≠ 0, отчёт = 0 | ложное заполнение |
| отчёт ≠ 0, в источнике нет строки | пропуск |
| отчёт ≠ 0, источник = 0 | пропуск (строка есть, сумма 0) |


In [ ]:
def classify_row(excel_nz, src_present, src_nz):
    if excel_nz and src_nz:
        return 'оба ≠ 0'
    if (not excel_nz) and (not src_nz):
        return 'оба 0 / нет строки'
    if (not excel_nz) and src_nz:
        return 'источник ≠ 0, отчёт = 0'
    if excel_nz and not src_present:
        return 'отчёт ≠ 0, в источнике нет строки'
    return 'отчёт ≠ 0, источник = 0'


def score_source(month, name, excel_df, src_df, src_col):
    n_ex = len(excel_df)
    n_ex_nz = int(excel_df['excel_nz'].sum())
    n_ex_z = int((~excel_df['excel_nz']).sum())
    t = excel_df.merge(
        src_df[['inn_key', 'agr_id_key', src_col]].copy(),
        on=['inn_key', 'agr_id_key'],
        how='left',
    )
    t['src_amt'] = pd.to_numeric(t[src_col], errors='coerce')
    t['src_present'] = t['src_amt'].notna()
    t['src_nz'] = is_nz(t['src_amt'])
    t['bucket'] = [
        classify_row(bool(ez), bool(sp), bool(sz))
        for ez, sp, sz in zip(t['excel_nz'], t['src_present'], t['src_nz'])
    ]
    t['excel_gross'] = t['commission_excel'] * VAT
    both = t.loc[t['bucket'] == 'оба ≠ 0'].copy()
    if len(both):
        both['exact_net'] = np.isclose(both['src_amt'].fillna(0), both['commission_excel'].fillna(0), atol=EPS, rtol=0)
        both['exact_vat'] = np.isclose(both['src_amt'].fillna(0), both['excel_gross'].fillna(0), atol=EPS, rtol=0)
        t = t.merge(both[['inn_key', 'agr_id_key', 'exact_net', 'exact_vat']], on=['inn_key', 'agr_id_key'], how='left')
    else:
        t['exact_net'] = False
        t['exact_vat'] = False

    vc = t['bucket'].value_counts()
    agree = int(vc.get('оба ≠ 0', 0) + vc.get('оба 0 / нет строки', 0))
    fp = t.loc[t['bucket'] == 'источник ≠ 0, отчёт = 0']
    fn_abs = t.loc[t['bucket'] == 'отчёт ≠ 0, в источнике нет строки']
    fn_z = t.loc[t['bucket'] == 'отчёт ≠ 0, источник = 0']
    fn = pd.concat([fn_abs, fn_z], ignore_index=True)
    rec = {
        'report_month': month,
        'source': name,
        'n_excel': n_ex,
        'excel_nz': n_ex_nz,
        'excel_zero': n_ex_z,
        'excel_nz_pct': round(100.0 * n_ex_nz / n_ex, 2) if n_ex else np.nan,
        'agree_n': agree,
        'agree_pct': round(100.0 * agree / n_ex, 2) if n_ex else np.nan,
        'both_nz': int(vc.get('оба ≠ 0', 0)),
        'both_empty': int(vc.get('оба 0 / нет строки', 0)),
        'fp_src_nz_excel_zero': int(len(fp)),
        'fp_pct_of_excel': round(100.0 * len(fp) / n_ex, 2) if n_ex else np.nan,
        'fp_pct_of_excel_zero': round(100.0 * len(fp) / n_ex_z, 2) if n_ex_z else np.nan,
        'fp_sum_src': float(fp['src_amt'].fillna(0).sum()),
        'fn_excel_nz_src_absent': int(len(fn_abs)),
        'fn_excel_nz_src_zero': int(len(fn_z)),
        'fn_total': int(len(fn)),
        'fn_pct_of_excel_nz': round(100.0 * len(fn) / n_ex_nz, 2) if n_ex_nz else np.nan,
        'fn_sum_excel': float(fn['commission_excel'].fillna(0).sum()),
        'recall_excel_nz': round(100.0 * int(vc.get('оба ≠ 0', 0)) / n_ex_nz, 2) if n_ex_nz else np.nan,
        'precision_src_nz': round(100.0 * int(vc.get('оба ≠ 0', 0)) / int(t['src_nz'].sum()), 2) if int(t['src_nz'].sum()) else np.nan,
        'exact_net_pct': round(100.0 * both['exact_net'].mean(), 2) if len(both) else np.nan,
        'exact_vat_pct': round(100.0 * both['exact_vat'].mean(), 2) if len(both) else np.nan,
        'sum_excel_both_nz': float(both['commission_excel'].sum()) if len(both) else 0.0,
        'sum_src_both_nz': float(both['src_amt'].sum()) if len(both) else 0.0,
        'sum_excel_nz': float(t.loc[t['excel_nz'], 'commission_excel'].sum()),
    }
    best_amt = 'net' if (rec['exact_net_pct'] or 0) >= (rec['exact_vat_pct'] or 0) else 'vat_x1.22'
    rec['best_amount_scale'] = best_amt
    rec['best_amount_exact_pct'] = rec['exact_net_pct'] if best_amt == 'net' else rec['exact_vat_pct']
    rec['disagree_n'] = rec['fp_src_nz_excel_zero'] + rec['fn_total']
    rec['disagree_pct'] = round(100.0 * rec['disagree_n'] / n_ex, 2) if n_ex else np.nan
    t['report_month'] = month
    t['source'] = name
    return t, rec


score_rows = []
tri_parts = []
buck_parts = []
bucket_order = [
    'оба ≠ 0',
    'оба 0 / нет строки',
    'источник ≠ 0, отчёт = 0',
    'отчёт ≠ 0, в источнике нет строки',
    'отчёт ≠ 0, источник = 0',
]

for month in FOCUS_MONTHS:
    ex_m = ex_all.loc[ex_all['report_month'] == month].copy()
    mpos_m = mpos_map.loc[mpos_map['report_month'] == month]
    cft_m = cft_map.loc[cft_map['report_month'] == month]
    mpos_tri, mpos_score = score_source(month, 'MPOS_RENT n_amt', ex_m, mpos_m, 'commission_mpos')
    cft_tri, cft_score = score_source(month, 'CFT o.c_calc_summ', ex_m, cft_m, 'commission_monthly_cft')
    score_rows.extend([mpos_score, cft_score])
    tri_parts.extend([mpos_tri, cft_tri])
    for name, tri in [('MPOS_RENT', mpos_tri), ('CFT c_calc_summ', cft_tri)]:
        g = (
            tri.groupby(['report_month', 'bucket'], as_index=False)
            .agg(keys=('agr_id_key', 'nunique'), inn=('inn_key', 'nunique'),
                 sum_excel=('commission_excel', 'sum'), sum_src=('src_amt', 'sum'))
        )
        g['source'] = name
        g['pct_excel'] = (100.0 * g['keys'] / len(ex_m)).round(2)
        buck_parts.append(g)

scores = pd.DataFrame(score_rows)
tri_all = pd.concat(tri_parts, ignore_index=True)
buck = pd.concat(buck_parts, ignore_index=True)
buck['bucket'] = pd.Categorical(buck['bucket'], categories=bucket_order, ordered=True)
buck = buck.sort_values(['report_month', 'source', 'bucket'])

print('=== 1) СХОДИТСЯ ===')
display(scores[[
    'report_month', 'source', 'n_excel', 'excel_nz', 'excel_nz_pct',
    'agree_n', 'agree_pct', 'both_nz', 'both_empty',
    'recall_excel_nz', 'precision_src_nz',
    'exact_net_pct', 'exact_vat_pct', 'best_amount_scale', 'best_amount_exact_pct',
]])

print('\\n=== 2) НЕ СХОДИТСЯ ===')
display(scores[[
    'report_month', 'source', 'disagree_n', 'disagree_pct',
    'fp_src_nz_excel_zero', 'fp_pct_of_excel', 'fp_pct_of_excel_zero', 'fp_sum_src',
    'fn_total', 'fn_excel_nz_src_absent', 'fn_excel_nz_src_zero',
    'fn_pct_of_excel_nz', 'fn_sum_excel',
]])

print('\\n=== Бакеты ===')
display(buck)


## 5) VERDICT за три летних месяца


In [ ]:
def pick_winner(a, b, key, higher=True):
    va, vb = a[key], b[key]
    if pd.isna(va) and pd.isna(vb):
        return None
    if pd.isna(va):
        return b['source']
    if pd.isna(vb):
        return a['source']
    if va == vb:
        return None
    if higher:
        return a['source'] if va > vb else b['source']
    return a['source'] if va < vb else b['source']


def month_verdict(month):
    mpos_score = scores.loc[(scores['report_month'] == month) & scores['source'].str.startswith('MPOS')].iloc[0]
    cft_score = scores.loc[(scores['report_month'] == month) & scores['source'].str.startswith('CFT')].iloc[0]
    w_agree = pick_winner(mpos_score, cft_score, 'agree_pct', True)
    w_fp = pick_winner(mpos_score, cft_score, 'fp_pct_of_excel', False)
    w_fn = pick_winner(mpos_score, cft_score, 'fn_pct_of_excel_nz', False)
    w_amt = pick_winner(mpos_score, cft_score, 'best_amount_exact_pct', True)
    mpos_pts = sum(1 for w in (w_agree, w_fp, w_fn, w_amt) if w and str(w).startswith('MPOS'))
    cft_pts = sum(1 for w in (w_agree, w_fp, w_fn, w_amt) if w and str(w).startswith('CFT'))
    if mpos_pts > cft_pts:
        winner = 'MPOS_RENT (n_amt)'
    elif cft_pts > mpos_pts:
        winner = 'ЦФТ (o.c_calc_summ)'
    else:
        winner = 'ничья'
    return mpos_score, cft_score, winner, mpos_pts, cft_pts, w_agree, w_fp, w_fn, w_amt


print('=' * 76)
print('VERDICT: commission_monthly vs отчёты июня–августа')
print('=' * 76)
month_winners = []
for month in FOCUS_MONTHS:
    mpos_score, cft_score, winner, mpos_pts, cft_pts, w_agree, w_fp, w_fn, w_amt = month_verdict(month)
    month_winners.append({'report_month': month, 'winner': winner, 'mpos_pts': mpos_pts, 'cft_pts': cft_pts})
    print(f'\\n--- {month} ---')
    print(
        f"Отчёт: {int(mpos_score['n_excel']):,} | ≠0 {int(mpos_score['excel_nz']):,} "
        f"({mpos_score['excel_nz_pct']:.2f}%) | =0 {int(mpos_score['excel_zero']):,}"
    )
    print('1) Сходится (присутствие)')
    for rec in (mpos_score, cft_score):
        print(
            f"   {rec['source']}: {int(rec['agree_n']):,}/{int(rec['n_excel']):,} = {rec['agree_pct']:.2f}%"
            f" | оба≠0 {int(rec['both_nz']):,} | оба пусто {int(rec['both_empty']):,}"
            f" | recall {rec['recall_excel_nz']:.2f}% | precision {rec['precision_src_nz']:.2f}%"
        )
    print(
        f"   сумма оба≠0: MPOS exact net {mpos_score['exact_net_pct']}% / vat {mpos_score['exact_vat_pct']}%"
        f" | ЦФТ exact net {cft_score['exact_net_pct']}% / vat {cft_score['exact_vat_pct']}%"
    )
    print('2) Не сходится')
    print(
        f"   источник≠0, отчёт=0: MPOS {int(mpos_score['fp_src_nz_excel_zero']):,} / {mpos_score['fp_sum_src']:,.0f} ₽"
        f" | ЦФТ {int(cft_score['fp_src_nz_excel_zero']):,} / {cft_score['fp_sum_src']:,.0f} ₽"
    )
    print(
        f"   отчёт≠0, источник пуст/0: MPOS {int(mpos_score['fn_total']):,} / {mpos_score['fn_sum_excel']:,.0f} ₽"
        f" | ЦФТ {int(cft_score['fn_total']):,} / {cft_score['fn_sum_excel']:,.0f} ₽"
    )
    print(f'   ИТОГ месяца: {winner}  (очки MPOS {mpos_pts} : ЦФТ {cft_pts})')

# Свод за лето: сумма ключей
print('\\n' + '=' * 76)
print('СВОД ИЮНЬ–АВГУСТ')
print('=' * 76)
summer = (
    scores.groupby('source', as_index=False)
    .agg(
        n_excel=('n_excel', 'sum'),
        excel_nz=('excel_nz', 'sum'),
        agree_n=('agree_n', 'sum'),
        both_nz=('both_nz', 'sum'),
        fp_src_nz_excel_zero=('fp_src_nz_excel_zero', 'sum'),
        fp_sum_src=('fp_sum_src', 'sum'),
        fn_total=('fn_total', 'sum'),
        fn_sum_excel=('fn_sum_excel', 'sum'),
        sum_excel_nz=('sum_excel_nz', 'sum'),
    )
)
summer['agree_pct'] = (100.0 * summer['agree_n'] / summer['n_excel']).round(2)
summer['fp_pct'] = (100.0 * summer['fp_src_nz_excel_zero'] / summer['n_excel']).round(2)
summer['fn_pct_of_nz'] = (100.0 * summer['fn_total'] / summer['excel_nz']).round(2)
display(summer)

mpos_s = summer.loc[summer['source'].str.startswith('MPOS')].iloc[0]
cft_s = summer.loc[summer['source'].str.startswith('CFT')].iloc[0]
summer_winner = 'MPOS_RENT (n_amt)' if mpos_s['agree_pct'] >= cft_s['agree_pct'] and mpos_s['fp_pct'] <= cft_s['fp_pct'] else (
    'ЦФТ (o.c_calc_summ)' if cft_s['agree_pct'] > mpos_s['agree_pct'] else 'MPOS_RENT (n_amt)'
)
print()
print(f"Лето, присутствие: MPOS {mpos_s['agree_pct']}% ({int(mpos_s['agree_n']):,}/{int(mpos_s['n_excel']):,})"
      f" | ЦФТ {cft_s['agree_pct']}% ({int(cft_s['agree_n']):,}/{int(cft_s['n_excel']):,})")
print(f"Лето, ложные заполнения: MPOS {int(mpos_s['fp_src_nz_excel_zero']):,} / {mpos_s['fp_sum_src']:,.0f} ₽"
      f" | ЦФТ {int(cft_s['fp_src_nz_excel_zero']):,} / {cft_s['fp_sum_src']:,.0f} ₽")
print(f"Лето, пропуски ненулей отчёта: MPOS {int(mpos_s['fn_total']):,} / {mpos_s['fn_sum_excel']:,.0f} ₽"
      f" | ЦФТ {int(cft_s['fn_total']):,} / {cft_s['fn_sum_excel']:,.0f} ₽")
print(f'ИТОГ за три месяца: точнее для commission_monthly → {summer_winner}')
print('Победители по месяцам:', month_winners)
print('=' * 76)

print('\\n=== TOP расхождений по месяцам (до 8 строк) ===')
for month in FOCUS_MONTHS:
    print(f'\\n{month} | источник ≠ 0, отчёт = 0')
    for src in ['MPOS_RENT n_amt', 'CFT o.c_calc_summ']:
        top = (
            tri_all.loc[(tri_all['report_month'] == month) & (tri_all['source'] == src) & (tri_all['bucket'] == 'источник ≠ 0, отчёт = 0')]
            .assign(abs_src=lambda d: d['src_amt'].abs())
            .sort_values('abs_src', ascending=False)
            [['inn_key', 'agr_id_key', 'tariff', 'commission_excel', 'src_amt']]
            .head(8)
        )
        print(src, 'rows=', len(top))
        display(top)
    print(f'{month} | отчёт ≠ 0, источник пуст/0')
    for src in ['MPOS_RENT n_amt', 'CFT o.c_calc_summ']:
        top = (
            tri_all.loc[
                (tri_all['report_month'] == month) & (tri_all['source'] == src)
                & tri_all['bucket'].isin(['отчёт ≠ 0, в источнике нет строки', 'отчёт ≠ 0, источник = 0'])
            ]
            .sort_values('commission_excel', ascending=False)
            [['inn_key', 'agr_id_key', 'tariff', 'commission_excel', 'src_amt', 'bucket']]
            .head(8)
        )
        print(src)
        display(top)


## 6) Выгрузка


In [ ]:
out = OUT_DIR / 'summer_commission_monthly_winner_2026_06_08.xlsx'
fp = tri_all.loc[tri_all['bucket'] == 'источник ≠ 0, отчёт = 0'].copy()
fn = tri_all.loc[tri_all['bucket'].isin(['отчёт ≠ 0, в источнике нет строки', 'отчёт ≠ 0, источник = 0'])].copy()
with pd.ExcelWriter(out, engine='openpyxl') as w:
    excel_stat.to_excel(w, sheet_name='excel_stat', index=False)
    scores.to_excel(w, sheet_name='scores', index=False)
    summer.to_excel(w, sheet_name='summer_totals', index=False)
    buck.to_excel(w, sheet_name='buckets', index=False)
    fp.to_excel(w, sheet_name='src_nz_excel_0', index=False)
    fn.to_excel(w, sheet_name='excel_nz_src_empty', index=False)
print('Saved:', out)
print('NOTEBOOK_REV', NOTEBOOK_REV, '| заключение в секции 5')
